In [2]:
!brew install ipmitool

==> Auto-updating Homebrew...
Adjust how often this is run with `$HOMEBREW_AUTO_UPDATE_SECS` or disable with
`$HOMEBREW_NO_AUTO_UPDATE=1`. Hide these hints with `$HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
==> Auto-updated Homebrew!
Updated 2 taps (homebrew/core and homebrew/cask).
==> New Formulae
adplay: Command-line player for OPL2 music
any2fasta: Convert various sequence formats to FASTA
av1an: Cross-platform command-line encoding framework
azurite: Lightweight server clone of Azure Storage that simulates it locally
beads: Memory upgrade for your coding agent
beads_viewer: Terminal-based UI for the Beads issue tracker
ccextractor: Tool for extracting closed captions from video files
cek: Explore the (overlay) filesystem and layers of OCI container images
classifier: Text classification with Bayesian, LSI, Logistic Regression, and kNN
codanna: Code intelligence system with semantic search
cronboard: Terminal-based dashboard for managing cron jobs locally and on servers
ctags-lsp: L

In [19]:
import subprocess
import re

def get_ipmi_temp(host, user, password, sensor_name="Temp"):
    # The command to get a specific sensor reading
    cmd = [
        "ipmitool", "-I", "lanplus", 
        "-H", host, 
        "-U", user, 
        "-P", password, 
        "-L", "User", "sdr", "type", "temperature", "|", "grep", "-v", "Disabled"
    ]

    try:
        # Run the command and capture output
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        
        # Output looks like: Temp             | 01h | ok  |  3.1 | 34 degrees C
        for line in result.stdout.splitlines():
            if "|" in line and sensor_name in line:
                parts = [p.strip() for p in line.split("|")]        
                if len(parts) >= 2:
                    return {
                        "sensor": parts[0],
                        "value": parts[4]
                    }
    except subprocess.CalledProcessError as e:
        print(f"Error communicating with iDRAC: {e.stderr}")
    except ValueError:
        print("Sensor returned 'na' or non-numeric value.")
    
    return None

# --- Configuration ---
IDRAC_IP = "192.168.22.222"
USER = "root"
PASS = "calvin"

# --- Run the Demo ---
data = get_ipmi_temp(IDRAC_IP, USER, PASS, "Inlet Temp")

if data:
    print(f"[{data['sensor']}] Current Reading: {data['value']}")
    
data = get_ipmi_temp(IDRAC_IP, USER, PASS, "Temp")

if data:
    print(f"[{data['sensor']}] Current Reading: {data['value']}")

[Inlet Temp] Current Reading: 28 degrees C
[Temp] Current Reading: 32 degrees C
